# Transformer: MLM Pretraining → Akkadian→English Machine Translation

**Stage 1 — MLM Pretraining (Pretext Task)**  
Train the Transformer encoder on `pretrain_large.csv` using Masked Language Modelling.
This teaches the encoder Akkadian semantics before it ever sees a translation pair.

**Stage 2 — Machine Translation (Downstream Task)**  
Attach a decoder to the pretrained encoder, then fine-tune the full model on
`data.csv` (transliteration → English translation pairs).

**Features:**
- ✅ **BPE Tokenisation** — 4000-token Akkadian vocab (with `<mask>`), 6000-token English vocab
- ✅ **BERT-style MLM** — 15% masking (80% `<mask>`, 10% random, 10% keep)
- ✅ **Same Vanilla Transformer** encoder architecture throughout both stages
- ✅ **Encoder weight transfer** — pretrained encoder loaded into the MT model
- ✅ **Warm-up fine-tuning** — encoder frozen for first N epochs, then unfrozen
- ✅ **Noam LR schedule**, **label smoothing**, **gradient clipping**, **early stopping**
- ✅ **Beam Search** decoding + BLEU / chrF++ evaluation

## Overall Pipeline

```
pretrain_large.csv  ──►  BPE tokenizer (4k Akkadian vocab)
                    ──►  MLM pretraining  →  pretrained encoder weights
                                                    │
data.csv  ──────────────────────────────────────────┘
                    ──►  Load pretrained encoder into TransformerMT
                    ──►  Fine-tune (transliteration → English)
                    ──►  Evaluate BLEU / chrF++
```

---
# STAGE 1 — MLM PRETRAINING
---

## 0. Imports & Seeds

In [ ]:
import sys
print(sys.executable)

In [ ]:
import math
import random
import heapq
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sacrebleu import corpus_bleu, corpus_chrf
from tqdm import tqdm

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Load & Clean Pretraining Data

In [ ]:
pretrain_df = pd.read_csv("pretrain_large.csv")
pretrain_df = pretrain_df[['transliteration']].dropna().reset_index(drop=True)

def clean_text(text: str) -> str:
    text = str(text).lower().strip()
    text = text.replace('"', '')
    text = text.replace('<gap>', ' <sep> ')
    return text

pretrain_df['transliteration'] = pretrain_df['transliteration'].apply(clean_text)
pretrain_df = pretrain_df[pretrain_df['transliteration'].str.len() >= 4].reset_index(drop=True)

print(f"Total pretraining samples: {len(pretrain_df):,}")
print("Sample:", pretrain_df['transliteration'].iloc[0])

## 2. Pretraining Train / Val Split

In [ ]:
pretrain_train_texts, pretrain_val_texts = train_test_split(
    pretrain_df['transliteration'].tolist(), test_size=0.05, random_state=SEED
)
print(f"Pretrain Train: {len(pretrain_train_texts):,} | Val: {len(pretrain_val_texts):,}")

## 3. BPE Tokenisation for Akkadian

We train a **single** Akkadian BPE tokeniser here on the large pretraining corpus.
This same tokeniser is reused in Stage 2 as `src_tokenizer` — so both stages
share identical Akkadian token IDs.

An extra special token `<mask>` is added for the MLM task.

In [ ]:
# Special tokens — same as MT notebook PLUS <mask> for MLM
SPECIAL_TOKENS = ["<pad>", "<unk>", "<sos>", "<eos>", "<sep>", "<mask>"]

def train_bpe(texts: list, vocab_size: int) -> Tokenizer:
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        min_frequency=2
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

print("Training Akkadian BPE tokenizer on pretrain_large.csv ...")
src_tokenizer = train_bpe(pretrain_train_texts, vocab_size=4000)

SRC_VOCAB_SIZE = src_tokenizer.get_vocab_size()
SRC_PAD_IDX    = src_tokenizer.token_to_id("<pad>")
SRC_UNK_IDX    = src_tokenizer.token_to_id("<unk>")
SRC_SOS_IDX    = src_tokenizer.token_to_id("<sos>")
SRC_EOS_IDX    = src_tokenizer.token_to_id("<eos>")
MASK_IDX       = src_tokenizer.token_to_id("<mask>")
MLM_SPECIAL_IDS = {SRC_PAD_IDX, SRC_UNK_IDX, SRC_SOS_IDX, SRC_EOS_IDX, MASK_IDX}

print(f"Source BPE vocab : {SRC_VOCAB_SIZE}")
print(f"PAD={SRC_PAD_IDX}  UNK={SRC_UNK_IDX}  SOS={SRC_SOS_IDX}  EOS={SRC_EOS_IDX}  MASK={MASK_IDX}")

# Sanity check
sample_ids = src_tokenizer.encode(pretrain_train_texts[0]).ids
print(f"\nSample  : '{pretrain_train_texts[0]}'")
print(f"BPE ids : {sample_ids[:12]} ...")
print(f"Decoded : '{src_tokenizer.decode(sample_ids)}'")

## 4. BERT-Style MLM Masking

For each sequence, **15%** of tokens are selected as masking candidates:

| Replacement    | Probability | Reason |
|----------------|-------------|--------|
| `<mask>` token | 80%         | Force prediction from context |
| Random token   | 10%         | Prevent over-reliance on `<mask>` |
| Unchanged      | 10%         | Provide correct signal at some positions |

Loss is computed **only at masked positions** — all other positions use `ignore_index=-100`.

In [ ]:
MASK_RATIO     = 0.15
MASK_TOKEN_P   = 0.80
RANDOM_TOKEN_P = 0.10
IGNORE_IDX     = -100    # CrossEntropyLoss ignore_index for non-masked positions


def apply_mlm_masking(
    token_ids: list,
    vocab_size: int,
    mask_idx: int,
    special_ids: set,
    mask_ratio: float = MASK_RATIO,
) -> tuple:
    """
    Apply BERT-style MLM masking to a list of BPE token IDs.

    Returns
    -------
    masked_ids : list[int] — encoder input (some tokens replaced)
    labels     : list[int] — original IDs at masked positions, IGNORE_IDX elsewhere
    """
    n          = len(token_ids)
    masked_ids = list(token_ids)
    labels     = [IGNORE_IDX] * n

    # Eligible positions: not special tokens
    eligible = [i for i, t in enumerate(token_ids) if t not in special_ids]

    if not eligible:
        return masked_ids, labels

    n_mask = max(1, int(round(len(eligible) * mask_ratio)))
    chosen = random.sample(eligible, min(n_mask, len(eligible)))

    for pos in chosen:
        labels[pos] = token_ids[pos]   # store original as label
        r = random.random()
        if r < MASK_TOKEN_P:
            masked_ids[pos] = mask_idx
        elif r < MASK_TOKEN_P + RANDOM_TOKEN_P:
            masked_ids[pos] = random.randint(len(special_ids), vocab_size - 1)
        # else: keep original (10%)

    return masked_ids, labels


# Sanity check
_ids = src_tokenizer.encode(pretrain_train_texts[0]).ids
_masked, _labels = apply_mlm_masking(_ids, SRC_VOCAB_SIZE, MASK_IDX, MLM_SPECIAL_IDS)
_n_masked = sum(1 for l in _labels if l != IGNORE_IDX)
print(f"Tokens: {len(_ids)} | Masked: {_n_masked} ({100*_n_masked/len(_ids):.1f}%)")

## 5. MLM Dataset & DataLoader

In [ ]:
MLM_MAX_LEN    = 80
MLM_BATCH_SIZE = 64


class AkkadianMLMDataset(Dataset):
    """
    Applies fresh random MLM masking on every __getitem__ call,
    so the model sees different corruptions across epochs.
    """
    def __init__(self, texts: list, tokenizer, mask_ratio: float = MASK_RATIO):
        self.mask_ratio = mask_ratio
        self.encoded = [
            tokenizer.encode(t).ids[:MLM_MAX_LEN]
            for t in texts
        ]
        self.encoded = [e for e in self.encoded if len(e) >= 4]

    def __len__(self):
        return len(self.encoded)

    def __getitem__(self, idx):
        token_ids = self.encoded[idx]
        masked_ids, labels = apply_mlm_masking(
            token_ids, SRC_VOCAB_SIZE, MASK_IDX, MLM_SPECIAL_IDS, self.mask_ratio
        )
        return (
            torch.tensor(masked_ids, dtype=torch.long),
            torch.tensor(labels,     dtype=torch.long)
        )


def mlm_collate(batch):
    masked_seqs, label_seqs = zip(*batch)
    masked_padded = pad_sequence(masked_seqs, batch_first=True, padding_value=SRC_PAD_IDX)
    labels_padded = pad_sequence(label_seqs,  batch_first=True, padding_value=IGNORE_IDX)
    return masked_padded, labels_padded


mlm_train_dataset = AkkadianMLMDataset(pretrain_train_texts, src_tokenizer)
mlm_val_dataset   = AkkadianMLMDataset(pretrain_val_texts,   src_tokenizer)

mlm_train_loader = DataLoader(
    mlm_train_dataset, batch_size=MLM_BATCH_SIZE, shuffle=True,
    collate_fn=mlm_collate, num_workers=2, pin_memory=True
)
mlm_val_loader = DataLoader(
    mlm_val_dataset, batch_size=MLM_BATCH_SIZE, shuffle=False,
    collate_fn=mlm_collate, num_workers=2, pin_memory=True
)

print(f"MLM Train samples : {len(mlm_train_dataset):,} | batches: {len(mlm_train_loader):,}")
print(f"MLM Val   samples : {len(mlm_val_dataset):,}   | batches: {len(mlm_val_loader):,}")

## 6. Shared Building Blocks — PositionalEncoding

Defined once here, reused by both `TransformerMLM` (Stage 1) and `TransformerMT` (Stage 2).

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (Vaswani et al., 2017).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)    # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## 7. Model — TransformerMLM (Stage 1)

```
Masked Akkadian BPE tokens
        ↓
  src_embedding  +  pos_encoder           ← same names as TransformerMT
        ↓
  transformer_encoder  [N layers]         ← same names as TransformerMT
        ↓
  MLM Head: LayerNorm → Linear → vocab logits
  Loss only at masked positions
```

Attribute names (`src_embedding`, `pos_encoder`, `transformer_encoder`) are
**identical** to those in `TransformerMT` so the weights transfer with `strict=False`.

In [ ]:
class TransformerMLM(nn.Module):
    """
    Transformer Encoder + MLM head for Akkadian pretraining.

    Parameters
    ----------
    vocab_size         : int   — Akkadian BPE vocab size
    d_model            : int   — embedding / model dimension  (default 256)
    nhead              : int   — number of attention heads    (default 8)
    num_encoder_layers : int   — encoder depth                (default 3)
    dim_feedforward    : int   — FFN hidden dim               (default 512)
    dropout            : float — dropout probability          (default 0.1)
    max_len            : int   — positional encoding max len  (default 512)
    pad_idx            : int   — padding token ID
    """
    def __init__(
        self,
        vocab_size:          int,
        d_model:             int   = 256,
        nhead:               int   = 8,
        num_encoder_layers:  int   = 3,
        dim_feedforward:     int   = 512,
        dropout:             float = 0.1,
        max_len:             int   = 512,
        pad_idx:             int   = 0,
    ):
        super().__init__()
        self.d_model  = d_model
        self.pad_idx  = pad_idx
        self.vocab_size = vocab_size

        # Named identically to TransformerMT for seamless weight transfer
        self.src_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder   = PositionalEncoding(d_model, dropout, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,    # Pre-LN — same as TransformerMT
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers,
            norm=nn.LayerNorm(d_model)
        )

        # MLM head: LayerNorm + Linear
        self.mlm_norm = nn.LayerNorm(d_model)
        self.mlm_head = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'embedding' in name:
                nn.init.normal_(p, mean=0.0, std=self.d_model ** -0.5)
            elif p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_key_padding_mask(self, src: torch.Tensor) -> torch.Tensor:
        return src == self.pad_idx    # True = pad position → ignore

    def encode(self, src: torch.Tensor) -> torch.Tensor:
        """src: (batch, seq_len) → encoder_output: (batch, seq_len, d_model)"""
        src_pad_mask = self.make_src_key_padding_mask(src)
        src_emb      = self.pos_encoder(
            self.src_embedding(src) * math.sqrt(self.d_model)
        )
        return self.transformer_encoder(src_emb, src_key_padding_mask=src_pad_mask)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """Returns logits: (batch, seq_len, vocab_size)"""
        enc_out = self.encode(src)
        return self.mlm_head(self.mlm_norm(enc_out))

    def get_encoder_state_dict(self) -> dict:
        """Return only the encoder weights (no MLM head) for transfer to TransformerMT."""
        encoder_prefixes = ('src_embedding', 'pos_encoder', 'transformer_encoder')
        return {
            k: v for k, v in self.state_dict().items()
            if any(k.startswith(p) for p in encoder_prefixes)
        }

## 8. Instantiate MLM Model, Optimiser, Scheduler & Loss

In [ ]:
# Hyperparameters — identical values used in Stage 2
D_MODEL         = 256
NHEAD           = 8
NUM_ENC_LAYERS  = 3
NUM_DEC_LAYERS  = 3
DIM_FEEDFORWARD = 512
DROPOUT         = 0.1
LABEL_SMOOTHING = 0.1
WARMUP_STEPS    = 4000

mlm_model = TransformerMLM(
    vocab_size          = SRC_VOCAB_SIZE,
    d_model             = D_MODEL,
    nhead               = NHEAD,
    num_encoder_layers  = NUM_ENC_LAYERS,
    dim_feedforward     = DIM_FEEDFORWARD,
    dropout             = DROPOUT,
    max_len             = 512,
    pad_idx             = SRC_PAD_IDX,
).to(device)

mlm_optimizer = optim.Adam(
    mlm_model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9
)

def noam_lambda(step: int) -> float:
    step = max(step, 1)
    return (D_MODEL ** -0.5) * min(step ** -0.5, step * WARMUP_STEPS ** -1.5)

mlm_scheduler = optim.lr_scheduler.LambdaLR(mlm_optimizer, lr_lambda=noam_lambda)

mlm_criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_IDX,
    label_smoothing=LABEL_SMOOTHING
)

total_params = sum(p.numel() for p in mlm_model.parameters() if p.requires_grad)
print(f"MLM model trainable parameters: {total_params:,}")
print(mlm_model)

## 9. MLM Evaluation Helper

In [ ]:
@torch.no_grad()
def evaluate_mlm(model, loader, criterion, device) -> tuple:
    """
    Compute average MLM loss and token accuracy over a DataLoader.
    Returns (avg_loss, accuracy) — accuracy computed only at masked positions.
    """
    model.eval()
    total_loss    = 0.0
    total_correct = 0
    total_masked  = 0

    for masked_inp, labels in loader:
        masked_inp = masked_inp.to(device)
        labels     = labels.to(device)

        logits      = model(masked_inp)                        # (batch, seq_len, vocab)
        flat_logits = logits.reshape(-1, SRC_VOCAB_SIZE)
        flat_labels = labels.reshape(-1)

        total_loss += criterion(flat_logits, flat_labels).item()

        mask    = flat_labels != IGNORE_IDX
        preds   = flat_logits.argmax(dim=-1)
        total_correct += (preds[mask] == flat_labels[mask]).sum().item()
        total_masked  += mask.sum().item()

    return total_loss / len(loader), total_correct / max(total_masked, 1)

## 10. MLM Training Loop

In [ ]:
MLM_N_EPOCHS  = 100
MLM_CLIP      = 1.0
MLM_PATIENCE  = 5
MLM_BEST_PATH = "transformer_akkadian_mlm_best.pt"

mlm_best_val_loss    = float('inf')
mlm_patience_counter = 0
mlm_history          = []
mlm_global_step      = 0

for epoch in range(MLM_N_EPOCHS):

    # ── Training ──────────────────────────────────────────────────────
    mlm_model.train()
    train_loss    = 0.0
    train_correct = 0
    train_masked  = 0

    for masked_inp, labels in tqdm(
        mlm_train_loader, desc=f"[MLM] Epoch {epoch+1}/{MLM_N_EPOCHS} [Train]"
    ):
        masked_inp = masked_inp.to(device)
        labels     = labels.to(device)

        mlm_optimizer.zero_grad()

        logits      = mlm_model(masked_inp)
        flat_logits = logits.reshape(-1, SRC_VOCAB_SIZE)
        flat_labels = labels.reshape(-1)

        loss = mlm_criterion(flat_logits, flat_labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), MLM_CLIP)
        mlm_optimizer.step()

        mlm_global_step += 1
        mlm_scheduler.step()

        train_loss += loss.item()
        mask    = flat_labels != IGNORE_IDX
        preds   = flat_logits.argmax(dim=-1)
        train_correct += (preds[mask] == flat_labels[mask]).sum().item()
        train_masked  += mask.sum().item()

    avg_train_loss = train_loss / len(mlm_train_loader)
    train_acc      = train_correct / max(train_masked, 1)

    # ── Validation ────────────────────────────────────────────────────
    avg_val_loss, val_acc = evaluate_mlm(mlm_model, mlm_val_loader, mlm_criterion, device)

    current_lr = mlm_optimizer.param_groups[0]['lr']

    mlm_history.append({
        'epoch': epoch + 1, 'train_loss': avg_train_loss,
        'train_acc': train_acc, 'val_loss': avg_val_loss,
        'val_acc': val_acc, 'lr': current_lr, 'step': mlm_global_step,
    })

    print(
        f"\n[MLM] Epoch {epoch+1} "
        f"| Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.4f} "
        f"| Val Loss: {avg_val_loss:.4f}  Acc: {val_acc:.4f} "
        f"| LR: {current_lr:.6f} | Step: {mlm_global_step:,}"
    )

    if avg_val_loss < mlm_best_val_loss:
        mlm_best_val_loss    = avg_val_loss
        mlm_patience_counter = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state': mlm_model.state_dict(),
            'val_loss': mlm_best_val_loss,
            'val_acc': val_acc,
        }, MLM_BEST_PATH)
        print(f"  ✅ New best MLM model saved  (val_loss={mlm_best_val_loss:.4f}  acc={val_acc:.4f})")
    else:
        mlm_patience_counter += 1
        print(f"  ⏳ No improvement ({mlm_patience_counter}/{MLM_PATIENCE})")
        if mlm_patience_counter >= MLM_PATIENCE:
            print("🛑 Early stopping triggered")
            break

print("\n✅ MLM Pretraining complete.")

## 11. MLM Training History

In [ ]:
mlm_history_df = pd.DataFrame(mlm_history)
print(mlm_history_df.to_string(index=False))

## 12. Load Best MLM Checkpoint & Extract Encoder Weights

In [ ]:
mlm_ckpt = torch.load(MLM_BEST_PATH, map_location=device)
mlm_model.load_state_dict(mlm_ckpt['model_state'])
print(f"Best MLM checkpoint loaded  (epoch={mlm_ckpt['epoch']}, val_loss={mlm_ckpt['val_loss']:.4f})")

# Extract encoder-only weights — these will be loaded into TransformerMT
pretrained_encoder_state = mlm_model.get_encoder_state_dict()
print(f"\nEncoder parameter tensors to transfer: {len(pretrained_encoder_state)}")
for k, v in pretrained_encoder_state.items():
    print(f"  {k:<55}  {tuple(v.shape)}")

---
# STAGE 2 — MACHINE TRANSLATION (DOWNSTREAM TASK)
---

## 13. Load & Clean MT Data

In [ ]:
mt_df = pd.read_csv("data.csv")
mt_df = mt_df[['transliteration', 'translation']].dropna()
mt_df = mt_df.rename(columns={'transliteration': 'source'})

mt_df['source']      = mt_df['source'].apply(clean_text)
mt_df['translation'] = mt_df['translation'].apply(clean_text)

print(f"Total MT samples: {len(mt_df):,}")

## 14. MT Train / Val / Test Split

Split **before** training the English BPE tokeniser to avoid data leakage. 80 / 10 / 10.

In [ ]:
train_df, temp_df = train_test_split(mt_df, test_size=0.2, random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"MT  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 15. English BPE Tokeniser

The Akkadian `src_tokenizer` was already trained in Stage 1 on `pretrain_large.csv`.
Here we train only the English target tokeniser on the MT training split.

In [ ]:
print("Training English BPE tokenizer...")
tgt_tokenizer = train_bpe(train_df['translation'].tolist(), vocab_size=6000)

TGT_VOCAB_SIZE = tgt_tokenizer.get_vocab_size()
TGT_PAD_IDX    = tgt_tokenizer.token_to_id("<pad>")
SOS_IDX        = tgt_tokenizer.token_to_id("<sos>")
EOS_IDX        = tgt_tokenizer.token_to_id("<eos>")

print(f"Source BPE vocab : {SRC_VOCAB_SIZE} (pretrained)")
print(f"Target BPE vocab : {TGT_VOCAB_SIZE}")

## 16. Encode MT Splits with BPE

In [ ]:
def encode_bpe(texts, tokenizer):
    return [enc.ids for enc in tokenizer.encode_batch(texts)]

for split_df in [train_df, val_df, test_df]:
    split_df['src_ids'] = encode_bpe(split_df['source'].tolist(), src_tokenizer)
    tgt_ids_raw = encode_bpe(split_df['translation'].tolist(), tgt_tokenizer)
    split_df['tgt_ids'] = [[SOS_IDX] + ids + [EOS_IDX] for ids in tgt_ids_raw]

sample = train_df.iloc[0]
print("Source  :", sample['source'])
print("BPE ids :", sample['src_ids'][:12], "...")
print("Decoded :", src_tokenizer.decode(sample['src_ids']))

## 17. Filter by Length

In [ ]:
def filter_by_length(df, max_len=80):
    df['src_len'] = df['src_ids'].apply(len)
    df['tgt_len'] = df['tgt_ids'].apply(len)
    df = df[(df['src_len'] <= max_len) & (df['tgt_len'] <= max_len)]
    df = df.sort_values(by='src_len').reset_index(drop=True)
    return df

train_df = filter_by_length(train_df, 80)
val_df   = filter_by_length(val_df,   80)
test_df  = filter_by_length(test_df,  80)

print(f"After filtering → Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 18. MT Dataset & DataLoader

In [ ]:
MT_BATCH_SIZE = 32


class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df['src_ids'].tolist()
        self.tgt = df['tgt_ids'].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])


def mt_collate_fn(batch):
    """Pad to longest sequence in batch; batch-first (batch, seq_len)."""
    src_seqs, tgt_seqs = zip(*batch)
    src_padded = pad_sequence(src_seqs, batch_first=True, padding_value=SRC_PAD_IDX)
    tgt_padded = pad_sequence(tgt_seqs, batch_first=True, padding_value=TGT_PAD_IDX)
    return src_padded, tgt_padded


mt_train_dataset = TranslationDataset(train_df)
mt_val_dataset   = TranslationDataset(val_df)
mt_test_dataset  = TranslationDataset(test_df)

mt_train_loader = DataLoader(mt_train_dataset, batch_size=MT_BATCH_SIZE, shuffle=False, collate_fn=mt_collate_fn)
mt_val_loader   = DataLoader(mt_val_dataset,   batch_size=MT_BATCH_SIZE, shuffle=False, collate_fn=mt_collate_fn)
mt_test_loader  = DataLoader(mt_test_dataset,  batch_size=MT_BATCH_SIZE, shuffle=False, collate_fn=mt_collate_fn)

print(f"MT Train batches: {len(mt_train_loader)} | Val: {len(mt_val_loader)} | Test: {len(mt_test_loader)}")

## 19. Model — TransformerMT (Stage 2)

```
SOURCE  →  src_embedding + pos_encoder   ← loaded from pretrained MLM encoder
        →  transformer_encoder [N]        ← loaded from pretrained MLM encoder
        →  memory  (batch, src_len, d_model)

TARGET  →  tgt_embedding + pos_decoder   ← randomly initialised
        →  transformer_decoder [N]        ← randomly initialised
        →  fc_out → tgt vocab logits
```

In [ ]:
class TransformerMT(nn.Module):
    """
    Encoder-Decoder Transformer for Akkadian → English translation.
    Encoder attribute names match TransformerMLM for direct weight transfer.

    Parameters
    ----------
    src_vocab_size     : int
    tgt_vocab_size     : int
    d_model            : int   (default 256)
    nhead              : int   (default 8)
    num_encoder_layers : int   (default 3)
    num_decoder_layers : int   (default 3)
    dim_feedforward    : int   (default 512)
    dropout            : float (default 0.1)
    max_len            : int   (default 512)
    src_pad_idx        : int
    tgt_pad_idx        : int
    """
    def __init__(
        self,
        src_vocab_size:      int,
        tgt_vocab_size:      int,
        d_model:             int   = 256,
        nhead:               int   = 8,
        num_encoder_layers:  int   = 3,
        num_decoder_layers:  int   = 3,
        dim_feedforward:     int   = 512,
        dropout:             float = 0.1,
        max_len:             int   = 512,
        src_pad_idx:         int   = 0,
        tgt_pad_idx:         int   = 0,
    ):
        super().__init__()
        self.d_model     = d_model
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx

        # ── Encoder side (names match TransformerMLM) ─────────────────
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=src_pad_idx)
        self.pos_encoder   = PositionalEncoding(d_model, dropout, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers, norm=nn.LayerNorm(d_model)
        )

        # ── Decoder side (randomly initialised) ───────────────────────
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=tgt_pad_idx)
        self.pos_decoder   = PositionalEncoding(d_model, dropout, max_len)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer, num_layers=num_decoder_layers, norm=nn.LayerNorm(d_model)
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'embedding' in name:
                nn.init.normal_(p, mean=0.0, std=self.d_model ** -0.5)
            elif p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_key_padding_mask(self, src):
        return src == self.src_pad_idx

    def make_tgt_key_padding_mask(self, tgt):
        return tgt == self.tgt_pad_idx

    def make_causal_mask(self, tgt_len, device):
        return torch.triu(torch.ones(tgt_len, tgt_len, device=device), diagonal=1).bool()

    def encode(self, src: torch.Tensor) -> torch.Tensor:
        src_pad_mask = self.make_src_key_padding_mask(src)
        src_emb      = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        return self.transformer_encoder(src_emb, src_key_padding_mask=src_pad_mask)

    def decode(self, tgt: torch.Tensor, memory: torch.Tensor, src: torch.Tensor) -> torch.Tensor:
        tgt_len         = tgt.size(1)
        tgt_causal_mask = self.make_causal_mask(tgt_len, tgt.device)
        tgt_pad_mask    = self.make_tgt_key_padding_mask(tgt)
        src_pad_mask    = self.make_src_key_padding_mask(src)
        tgt_emb         = self.pos_decoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        out = self.transformer_decoder(
            tgt_emb, memory,
            tgt_mask=tgt_causal_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )
        return self.fc_out(out)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        """
        src : (batch, src_len)
        tgt : (batch, tgt_len)  — includes <sos> and <eos>
        Decoder input  = tgt[:, :-1]   (<sos> … last token)
        Decoder labels = tgt[:,  1:]   (first token … <eos>)
        Returns logits: (batch, tgt_len-1, tgt_vocab)
        """
        memory = self.encode(src)
        return self.decode(tgt[:, :-1], memory, src)

## 20. Instantiate MT Model & Load Pretrained Encoder

In [ ]:
mt_model = TransformerMT(
    src_vocab_size     = SRC_VOCAB_SIZE,
    tgt_vocab_size     = TGT_VOCAB_SIZE,
    d_model            = D_MODEL,
    nhead              = NHEAD,
    num_encoder_layers = NUM_ENC_LAYERS,
    num_decoder_layers = NUM_DEC_LAYERS,
    dim_feedforward    = DIM_FEEDFORWARD,
    dropout            = DROPOUT,
    max_len            = 512,
    src_pad_idx        = SRC_PAD_IDX,
    tgt_pad_idx        = TGT_PAD_IDX,
).to(device)

# ── Load pretrained encoder weights ───────────────────────────────────
# strict=False: TransformerMT has extra keys (tgt_embedding, decoder, fc_out)
# that are not in pretrained_encoder_state — they stay at random init.
missing, unexpected = mt_model.load_state_dict(pretrained_encoder_state, strict=False)

# Verify: encoder keys should all be loaded (missing list should have NO encoder keys)
enc_missing = [k for k in missing if any(
    k.startswith(p) for p in ('src_embedding', 'pos_encoder', 'transformer_encoder')
)]
print(f"Encoder keys NOT loaded (should be empty) : {enc_missing}")
print(f"Unexpected keys         (should be empty) : {unexpected}")
print(f"Decoder keys randomly initialised         : {len(missing)} keys")
print("Pretrained encoder loaded into MT model ✅")

total_params = sum(p.numel() for p in mt_model.parameters() if p.requires_grad)
print(f"\nMT model trainable parameters: {total_params:,}")

## 21. MT Optimiser, Scheduler & Loss

**Warm-up fine-tuning strategy:**  
The encoder is **frozen for the first `FREEZE_EPOCHS` epochs** so the randomly initialised
decoder can catch up to the pretrained encoder representations before joint training begins.
After that, all parameters are unfrozen for full end-to-end fine-tuning.

In [ ]:
FREEZE_EPOCHS = 3    # freeze encoder for first N epochs

ENCODER_PARAM_NAMES = ('src_embedding', 'pos_encoder', 'transformer_encoder')

def freeze_encoder(model):
    for name, param in model.named_parameters():
        if any(name.startswith(p) for p in ENCODER_PARAM_NAMES):
            param.requires_grad = False
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Encoder FROZEN   | Frozen: {frozen:,} | Trainable: {trainable:,}")

def unfreeze_encoder(model):
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Encoder UNFROZEN | All trainable: {trainable:,}")

# Start with encoder frozen
freeze_encoder(mt_model)

mt_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, mt_model.parameters()),
    lr=1.0, betas=(0.9, 0.98), eps=1e-9
)

mt_scheduler = optim.lr_scheduler.LambdaLR(mt_optimizer, lr_lambda=noam_lambda)

mt_criterion = nn.CrossEntropyLoss(
    ignore_index=TGT_PAD_IDX,
    label_smoothing=LABEL_SMOOTHING
)

print(mt_model)

## 22. Inference Helpers

### 22a. Greedy Decode

In [ ]:
def translate_greedy(model, sentence, src_tokenizer, tgt_tokenizer, device, max_len=50):
    """Greedy auto-regressive decode for a single sentence."""
    model.eval()
    src_ids    = src_tokenizer.encode(sentence).ids
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)

    with torch.no_grad():
        memory = model.encode(src_tensor)

    generated = [SOS_IDX]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(generated).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model.decode(tgt_tensor, memory, src_tensor)
        next_id = logits[0, -1].argmax().item()
        if next_id == EOS_IDX:
            break
        generated.append(next_id)

    return tgt_tokenizer.decode(generated[1:])

### 22b. Beam Search Decode

In [ ]:
def translate_beam(
    model, sentence, src_tokenizer, tgt_tokenizer, device,
    beam_size=4, max_len=50, length_penalty=0.7
):
    """
    Beam search decoding for the Transformer.
    Maintains beam_size hypotheses and expands top candidates at each step.
    """
    model.eval()
    src_ids    = src_tokenizer.encode(sentence).ids
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)

    with torch.no_grad():
        memory = model.encode(src_tensor)

    beams     = [(0.0, [SOS_IDX])]
    completed = []

    for _ in range(max_len):
        all_candidates = []
        for log_prob, ids in beams:
            tgt_tensor = torch.tensor(ids).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model.decode(tgt_tensor, memory, src_tensor)
            log_probs = F.log_softmax(logits[0, -1], dim=-1)
            topk_lp, topk_idx = log_probs.topk(beam_size)

            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                new_log_prob = log_prob + lp
                new_ids      = ids + [idx]
                if idx == EOS_IDX:
                    pen_score = new_log_prob / (len(ids) ** length_penalty)
                    completed.append((pen_score, ids[1:]))
                else:
                    all_candidates.append((new_log_prob, new_ids))

        beams = sorted(all_candidates, key=lambda x: x[0], reverse=True)[:beam_size]
        if not beams:
            break

    if completed:
        best_ids = max(completed, key=lambda x: x[0])[1]
    else:
        best_ids = beams[0][1][1:]

    return tgt_tokenizer.decode(best_ids)

## 23. MT Evaluation Function

In [ ]:
def evaluate_mt(model, df_split, src_tokenizer, tgt_tokenizer, device,
                beam=True, beam_size=4):
    """Evaluate BLEU and chrF++ on a dataset split."""
    preds, refs = [], []

    decode_fn = (
        (lambda s: translate_beam(model, s, src_tokenizer, tgt_tokenizer, device, beam_size=beam_size))
        if beam
        else (lambda s: translate_greedy(model, s, src_tokenizer, tgt_tokenizer, device))
    )

    for _, row in df_split.iterrows():
        src  = row['source']
        ref  = row['translation'].replace("<sep>", " ").strip()
        pred = decode_fn(src).replace("<sep>", " ").strip()
        preds.append(pred)
        refs.append(ref)

    bleu = corpus_bleu(preds, [refs]).score
    chrf = corpus_chrf(preds, [refs]).score
    return bleu, chrf

## 24. MT Training Loop

- **Epochs 1 → `FREEZE_EPOCHS`**: encoder frozen, only decoder trains
- **Epochs `FREEZE_EPOCHS+1` → end**: full model fine-tuning
- Noam schedule steps every batch
- Best checkpoint on val loss; early stopping with `patience=10`

In [ ]:
MT_N_EPOCHS  = 300
MT_CLIP      = 1.0
MT_PATIENCE  = 20
MT_BEST_PATH = "transformer_akkadian_mt_best.pt"

mt_best_val_loss    = float('inf')
mt_patience_counter = 0
mt_history          = []
mt_global_step      = 0
encoder_unfrozen    = False

for epoch in range(MT_N_EPOCHS):

    # ── Unfreeze encoder after warm-up epochs ─────────────────────────
    if epoch == FREEZE_EPOCHS and not encoder_unfrozen:
        print(f"\n[MT] Epoch {epoch+1}: Unfreezing encoder for full fine-tuning...")
        unfreeze_encoder(mt_model)
        # Re-create optimiser so newly unfrozen params are included
        mt_optimizer = optim.Adam(
            mt_model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9
        )
        mt_scheduler = optim.lr_scheduler.LambdaLR(mt_optimizer, lr_lambda=noam_lambda)
        encoder_unfrozen = True

    # ── Training ──────────────────────────────────────────────────────
    mt_model.train()
    train_loss = 0.0

    for src, tgt in tqdm(mt_train_loader, desc=f"[MT] Epoch {epoch+1}/{MT_N_EPOCHS} [Train]"):
        src, tgt = src.to(device), tgt.to(device)

        mt_optimizer.zero_grad()

        # model.forward: decoder input = tgt[:,:-1], returns logits (batch, tgt_len-1, vocab)
        logits = mt_model(src, tgt)

        tgt_labels  = tgt[:, 1:].reshape(-1)
        logits_flat = logits.reshape(-1, TGT_VOCAB_SIZE)

        loss = mt_criterion(logits_flat, tgt_labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(mt_model.parameters(), MT_CLIP)
        mt_optimizer.step()

        mt_global_step += 1
        mt_scheduler.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(mt_train_loader)

    # ── Validation loss ───────────────────────────────────────────────
    mt_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt in mt_val_loader:
            src, tgt   = src.to(device), tgt.to(device)
            logits     = mt_model(src, tgt)
            tgt_flat   = tgt[:, 1:].reshape(-1)
            log_flat   = logits.reshape(-1, TGT_VOCAB_SIZE)
            val_loss  += mt_criterion(log_flat, tgt_flat).item()
    avg_val_loss = val_loss / len(mt_val_loader)

    # ── BLEU / chrF++ on val (greedy, fast) ───────────────────────────
    val_bleu, val_chrf = evaluate_mt(
        mt_model, val_df, src_tokenizer, tgt_tokenizer, device, beam=False
    )

    current_lr = mt_optimizer.param_groups[0]['lr']
    phase = "frozen-enc" if epoch < FREEZE_EPOCHS else "full-finetune"

    mt_history.append({
        'epoch': epoch + 1, 'phase': phase,
        'train_loss': avg_train_loss, 'val_loss': avg_val_loss,
        'val_bleu': val_bleu, 'val_chrf': val_chrf,
        'lr': current_lr, 'step': mt_global_step,
    })

    print(
        f"\n[MT] Epoch {epoch+1} [{phase}] "
        f"| Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
        f"| Val BLEU: {val_bleu:.2f} | Val chrF++: {val_chrf:.2f} "
        f"| LR: {current_lr:.6f} | Step: {mt_global_step}"
    )

    if avg_val_loss < mt_best_val_loss:
        mt_best_val_loss    = avg_val_loss
        mt_patience_counter = 0
        torch.save(mt_model.state_dict(), MT_BEST_PATH)
        print(f"  ✅ New best MT model saved (val_loss={mt_best_val_loss:.4f})")
    else:
        mt_patience_counter += 1
        print(f"  ⏳ No improvement ({mt_patience_counter}/{MT_PATIENCE})")
        if mt_patience_counter >= MT_PATIENCE:
            print("🛑 Early stopping triggered")
            break

print("\n✅ MT Fine-tuning complete.")

## 25. MT Training History

In [ ]:
mt_history_df = pd.DataFrame(mt_history)
print(mt_history_df.to_string(index=False))

## 26. Final Evaluation on Test Set

Load best checkpoint and evaluate with both greedy and beam search (beam_size=4).

In [ ]:
mt_model.load_state_dict(torch.load(MT_BEST_PATH, map_location=device))

# Greedy
test_bleu_greedy, test_chrf_greedy = evaluate_mt(
    mt_model, test_df, src_tokenizer, tgt_tokenizer, device, beam=False
)

# Beam search
test_bleu_beam, test_chrf_beam = evaluate_mt(
    mt_model, test_df, src_tokenizer, tgt_tokenizer, device, beam=True, beam_size=4
)

print("=" * 58)
print("  TEST SET RESULTS  (Transformer + MLM Pretraining + BPE)")
print("=" * 58)
print(f"  Greedy  — BLEU: {test_bleu_greedy:.2f}  chrF++: {test_chrf_greedy:.2f}")
print(f"  Beam-4  — BLEU: {test_bleu_beam:.2f}  chrF++: {test_chrf_beam:.2f}")
print("=" * 58)

## 27. Attention Visualisation

Visualise **cross-attention** weights from the last decoder layer — which source BPE tokens
each generated English token attends to.

In [ ]:
def get_cross_attention(model, sentence, src_tokenizer, tgt_tokenizer, device, max_len=50):
    """
    Run greedy decode and collect cross-attention weights from the
    last decoder layer (averaged over all heads).
    Returns (tgt_tokens, src_tokens, attn_matrix).
    """
    model.eval()
    src_ids    = src_tokenizer.encode(sentence).ids
    src_tokens = [src_tokenizer.id_to_token(i) for i in src_ids]
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)

    cross_attn_weights = []

    def hook_fn(module, input, output):
        cross_attn_weights.append(output[1].detach().cpu())

    last_decoder_layer = model.transformer_decoder.layers[-1]
    hook = last_decoder_layer.multihead_attn.register_forward_hook(hook_fn)

    with torch.no_grad():
        memory = model.encode(src_tensor)

    generated = [SOS_IDX]
    attn_rows  = []

    for _ in range(max_len):
        cross_attn_weights.clear()
        tgt_t = torch.tensor(generated).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model.decode(tgt_t, memory, src_tensor)
        attn_last = cross_attn_weights[0][0, -1, :len(src_ids)].numpy()
        next_id   = logits[0, -1].argmax().item()
        if next_id == EOS_IDX:
            break
        generated.append(next_id)
        attn_rows.append(attn_last)

    hook.remove()
    tgt_tokens  = [tgt_tokenizer.id_to_token(i) for i in generated[1:]]
    attn_matrix = np.array(attn_rows)
    return tgt_tokens, src_tokens, attn_matrix


def plot_attention(sentence, tgt_tokens, src_tokens, attn_matrix):
    fig, ax = plt.subplots(
        figsize=(max(8, len(src_tokens)), max(6, len(tgt_tokens) // 2))
    )
    im = ax.matshow(attn_matrix, cmap='viridis')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(tgt_tokens)))
    ax.set_xticklabels(src_tokens, rotation=90, fontsize=9)
    ax.set_yticklabels(tgt_tokens, fontsize=9)
    ax.set_xlabel('Source (Akkadian BPE tokens)')
    ax.set_ylabel('Target (English BPE tokens)')
    ax.set_title(f'Cross-Attention (last decoder layer): "{sentence[:60]}"')
    plt.tight_layout()
    plt.savefig('transformer_mlm_mt_attention_map.png', dpi=150)
    plt.show()


sample_sentence = test_df.iloc[0]['source']
tgt_tok, src_tok, attn = get_cross_attention(
    mt_model, sample_sentence, src_tokenizer, tgt_tokenizer, device
)
plot_attention(sample_sentence, tgt_tok, src_tok, attn)

## 28. Sample Translations

In [ ]:
print("=" * 70)
print("  SAMPLE TRANSLATIONS  (Beam-4 | MLM-pretrained Transformer)")
print("=" * 70)

for i in range(min(5, len(test_df))):
    row  = test_df.iloc[i]
    src  = row['source']
    ref  = row['translation'].replace("<sep>", " ").strip()
    pred = translate_beam(
        mt_model, src, src_tokenizer, tgt_tokenizer, device, beam_size=4
    ).replace("<sep>", " ").strip()

    print(f"\n[{i+1}]")
    print(f"  SRC  : {src}")
    print(f"  REF  : {ref}")
    print(f"  PRED : {pred}")

print("=" * 70)